# Vessel Measurement — Structured Analytical Workflow

This notebook demonstrates a small, reproducible workflow for turning
pixel-based observations from overhead imagery into structured vessel
dimension estimates.

Steps:
1. Load measurement observations
2. Convert pixel distances to metres
3. Calculate the length-to-beam (L/B) ratio
4. Record measurement uncertainty
5. Export structured results

This is intentionally a foundational tool, not a machine-learning
pipeline. See `methodology/dimension-estimation.md` for the underlying
reasoning.

In [1]:
import sys
sys.path.append("../src")

import csv
from pathlib import Path

from measurement import measurement_with_uncertainty, length_to_beam_ratio, Measurement

print("Imports OK")

Imports OK


## 1. Load measurement observations

Each record represents one overhead-imagery observation: a pixel distance for length and beam, plus the image's Ground Sampling Distance (GSD).

In [2]:
observations = [
    {
        "vessel_id": "V-01",
        "broad_type": "tanker",
        "length_px": 364,
        "beam_px": 58,
        "gsd_m_per_px": 0.5,
    },
    {
        "vessel_id": "V-02",
        "broad_type": "lpg_carrier",
        "length_px": 372,
        "beam_px": 91,
        "gsd_m_per_px": 0.5,
    },
    {
        "vessel_id": "V-03",
        "broad_type": "container",
        "length_px": 596,
        "beam_px": 65,
        "gsd_m_per_px": 0.5,
    },
]

observations

[{'vessel_id': 'V-01',
  'broad_type': 'tanker',
  'length_px': 364,
  'beam_px': 58,
  'gsd_m_per_px': 0.5},
 {'vessel_id': 'V-02',
  'broad_type': 'lpg_carrier',
  'length_px': 372,
  'beam_px': 91,
  'gsd_m_per_px': 0.5},
 {'vessel_id': 'V-03',
  'broad_type': 'container',
  'length_px': 596,
  'beam_px': 65,
  'gsd_m_per_px': 0.5}]

## 2. Convert pixel distances to metres, with uncertainty

`measurement_with_uncertainty()` applies the GSD to both the pixel distance and a fixed pixel-level uncertainty margin (default 3 px).

In [3]:
results = []

for obs in observations:
    length = measurement_with_uncertainty(obs["length_px"], obs["gsd_m_per_px"])
    beam = measurement_with_uncertainty(obs["beam_px"], obs["gsd_m_per_px"])

    results.append({
        "vessel_id": obs["vessel_id"],
        "broad_type": obs["broad_type"],
        "length": length,
        "beam": beam,
    })

for r in results:
    print(f"{r['vessel_id']:>6} | {r['broad_type']:<12} | LOA {r['length']} | BEAM {r['beam']}")

  V-01 | tanker       | LOA 182.0 ± 1.5 m | BEAM 29.0 ± 1.5 m
  V-02 | lpg_carrier  | LOA 186.0 ± 1.5 m | BEAM 45.5 ± 1.5 m
  V-03 | container    | LOA 298.0 ± 1.5 m | BEAM 32.5 ± 1.5 m


## 3. Calculate L/B ratio

L/B ratio is one input among several used to support (never single-handedly determine) a classification hypothesis.

In [4]:
for r in results:
    ratio = length_to_beam_ratio(r["length"].value_m, r["beam"].value_m)
    r["lb_ratio"] = round(ratio, 2)
    print(f"{r['vessel_id']:>6} | L/B ratio: {r['lb_ratio']}")

  V-01 | L/B ratio: 6.28
  V-02 | L/B ratio: 4.09
  V-03 | L/B ratio: 9.17


## 4. Record uncertainty explicitly

Uncertainty is never dropped from the record — it travels with the estimate through export, exactly as it should in an intelligence product.

In [5]:
for r in results:
    print(
        f"{r['vessel_id']:>6} | LOA {r['length'].value_m} ± {r['length'].uncertainty_m} m"
        f" | BEAM {r['beam'].value_m} ± {r['beam'].uncertainty_m} m"
        f" | L/B {r['lb_ratio']}"
    )

  V-01 | LOA 182.0 ± 1.5 m | BEAM 29.0 ± 1.5 m | L/B 6.28
  V-02 | LOA 186.0 ± 1.5 m | BEAM 45.5 ± 1.5 m | L/B 4.09
  V-03 | LOA 298.0 ± 1.5 m | BEAM 32.5 ± 1.5 m | L/B 9.17


## 5. Export structured results

Exporting to CSV mirrors the structure used in `data/vessel-assessments.csv`, keeping dimension estimates and their uncertainty machine-readable.

In [6]:
output_path = Path("vessel-measurement-export.csv")

with output_path.open("w", newline="") as f:
    writer = csv.writer(f)
    writer.writerow([
        "vessel_id", "broad_type",
        "estimated_length_m", "length_uncertainty_m",
        "estimated_beam_m", "beam_uncertainty_m",
        "lb_ratio",
    ])
    for r in results:
        writer.writerow([
            r["vessel_id"], r["broad_type"],
            r["length"].value_m, r["length"].uncertainty_m,
            r["beam"].value_m, r["beam"].uncertainty_m,
            r["lb_ratio"],
        ])

print(f"Exported {len(results)} records to {output_path}")

Exported 3 records to vessel-measurement-export.csv


## Notes

- Dimension estimates use a fixed 3-pixel uncertainty margin for
  illustration. In a production workflow, uncertainty should also
  account for vessel orientation relative to the image, wake and
  shadow contamination, and georeferencing accuracy.
- L/B ratio supports — it does not by itself establish — a
  classification hypothesis. See
  `methodology/evidence-vs-inference.md`.